<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Gymnasiet_Lab_2_Maskininlarning/Lektion_5_Bedragerier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💳 Maskininlärning – Lektion 5: Kreditkortsbedrägerier och obalanserad data

**Målgrupp:** Gymnasiet, 16 år, inga förkunskaper krävs  
**Tid:** ca 45 minuter  
**Mål:** Förstå problemet med obalanserad data och varför noggrannhet ibland är ett missvisande mått

---

### Upphovspersoner
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se  
Gymnasieversion baserad på originalverket ovan.

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/

---
## 🌍 Del 1 – Från blommor till verkligheten

Vi har jobbat med att klassificera iris-blommor – det var ett bra sätt att lära sig!  
Men nu tar vi ett steg mot ett **riktigt problem** som AI-ingenjörer jobbar med varje dag:

### Kreditkortsbedrägerier!

Varje dag görs **miljontals** kortbetalningar världen över.  
De flesta är normala köp – men en liten del är **bedrägerier (Fraud)**:
- Tjuvar som stjäl ett kortnummer och handlar utan ägarens vetskap
- Falska köp online
- Identitetsstöld

Banker vill ha en AI som **automatiskt** kan stoppa bedrägerier i realtid –  
redan innan banken godkänner betalningen!

```
Du klickar "Betala" på nätet
          │
          ▼
AI analyserar köpet på 0.1 sekunder
          │
    ┌─────┴──────┐
    │             │
 NORMALT     BEDRÄGERI?
 ✅ Godkänd   🚨 Stoppad!
```

---
## ⚠️ Del 2 – Det stora problemet: Obalanserad data

Här kommer det knepiga:

Av alla kreditkortsköp i världen är ungefär **99.8% normala** och bara **0.2% bedrägerier**.

Det kallas **Obalanserad data (Imbalanced Data)** – en kategori är extremt mycket vanligare.

### Den bedrägliga statistiken! 🎭

Föreställ dig denna AI-modell:

```python
def min_super_ai(köp):
    return "INTE BEDRÄGERI"   # Alltid samma svar!
```

Hur bra är den här modellen? Om vi mäter **Noggrannhet (Accuracy)**:

> Av 1000 köp är 998 normala och 2 bedrägerier.  
> Om vi gissar "inte bedrägeri" på **allt** → vi har rätt 998/1000 = **99.8% noggrannhet!**

**Men modellen hittar NOLL bedrägerier! Den är helt värdelös!** 🤦

> 🎯 **Lärdom:** Hög noggrannhet (Accuracy) garanterar INTE att modellen är bra.  
> Speciellt inte när data är obalanserad!

### 💬 Reflektionsfråga 5.1

En läkare testar ett cancertest.  
I gruppen som testas har 1% cancer och 99% inte cancer.

Testet säger "ingen cancer" på alla patienter och får 99% noggrannhet.

**Fråga:** Är det ett bra test? Vad är det egentliga problemet?

**Fråga 2:** Vad är värre för en patient – att testet missar en riktig cancer,  
eller att det felaktigt flaggar en frisk person som sjuk?  
*(Det finns inget universellt rätt svar – det beror på kontext!)*

---
## 📥 Del 3 – Ladda bedrägeridatan

Vi ska använda ett riktigt dataset med kreditkortstransaktioner.  
Datan hämtas från internet – det kan ta en liten stund att ladda!

> **OBS:** Kolumnerna heter V1, V2, ..., V28 av integritetsskäl –  
> de riktiga kolumnnamnen (t.ex. "butik", "belopp", "land") är dolda för att skydda kundernas integritet.  
> Kolumnen `Class` är det vi vill förutsäga: `0` = normalt köp, `1` = bedrägeri.

In [ ]:
!pip install xgboost -q
print("✅ xgboost installerat!")

In [ ]:
import pandas as pd

print("Laddar data från internet... (kan ta 30-60 sekunder)")

url = 'https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/refs/heads/master/creditcard.csv'
df = pd.read_csv(url)

print(f"✅ Data laddad!")
print(f"Antal transaktioner: {len(df):,}")
print(f"Antal kolumner: {len(df.columns)}")

---
## 🔍 Del 4 – Utforska obalansen

Nu ska vi se hur obalanserad datan verkligen är:

In [ ]:
# Räkna normala köp vs bedrägerier
antal_normala  = (df['Class'] == 0).sum()
antal_bedragerier = (df['Class'] == 1).sum()
totalt = len(df)

print(f"Normala transaktioner:  {antal_normala:>7,}  ({antal_normala/totalt:.2%})")
print(f"Bedrägerier:            {antal_bedragerier:>7,}  ({antal_bedragerier/totalt:.2%})")
print(f"Totalt:                 {totalt:>7,}")
print()
print(f"Det finns {antal_normala // antal_bedragerier} normala transaktioner för varje bedrägeri!")

### Visualisera obalansen

En bild säger mer än tusen ord:

In [ ]:
import matplotlib.pyplot as plt

etiketter = ['Normala köp', 'Bedrägerier']
antal = [antal_normala, antal_bedragerier]
farger = ['#4CAF50', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Stapeldiagram
axes[0].bar(etiketter, antal, color=farger)
axes[0].set_title('Antal transaktioner', fontsize=13)
axes[0].set_ylabel('Antal')
for i, v in enumerate(antal):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Kakdiagram
axes[1].pie(antal, labels=etiketter, colors=farger,
            autopct='%1.2f%%', startangle=90)
axes[1].set_title('Fördelning (%)', fontsize=13)

plt.suptitle('Obalanserad data (Imbalanced Data) – kreditkortstransaktioner',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🪤 Del 5 – Noggrannhetsfällan (Accuracy Paradox)

Låt oss bevisa problemet med ett experiment.  
Vi skapar en "dum" modell som alltid svarar "Inte bedrägeri":

In [ ]:
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import numpy as np

# Förbered data
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Den "dumma" modellen: gissa alltid 0 (inte bedrägeri)
y_pred_dum = pd.Series([0] * len(y_test))

noggrannhet_dum = accuracy_score(y_test, y_pred_dum)
print(f"Den 'dumma' modellens noggrannhet: {noggrannhet_dum:.2%}")
print()
print("Imponerande siffra – men modellen hittar NOLL bedrägerier!")
print(f"Antal missade bedrägerier: {(y_test == 1).sum()}")

---
## 🤖 Del 6 – Testa med en riktig AI-modell

Nu tränar vi en riktig modell och jämför:

In [ ]:
from xgboost import XGBClassifier

print("Tränar modellen... (kan ta 30-60 sekunder)")

modell = XGBClassifier(
    n_estimators=10,
    max_depth=3,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
modell.fit(X_train, y_train)

y_pred = modell.predict(X_test)

noggrannhet = accuracy_score(y_test, y_pred)
print(f"\n✅ Riktiga modellens noggrannhet: {noggrannhet:.2%}")

In [ ]:
# Rita förväxlingsmatrisen
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Den dumma modellen
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_dum,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False,
    ax=axes[0]
)
axes[0].set_title('"Dum" modell – gissar alltid Normalt', fontsize=11)

# Den riktiga modellen
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Normalt', 'Bedrägeri'],
    colorbar=False,
    ax=axes[1]
)
axes[1].set_title('Riktig AI-modell (XGBoost)', fontsize=11)

plt.suptitle('Jämförelse: Förväxlingsmatriser (Confusion Matrices)', fontsize=13)
plt.tight_layout()
plt.show()

### Tolka matriserna

Förväxlingsmatrisen för bedrägeriproblem har 4 rutor:

```
                    AI gissade:
                Normalt   Bedrägeri
Rätt  Normalt:  [ TN    |   FP   ]  ← Normala köp som AI hanterade rätt / fel
svar: Bedrägeri:[ FN    |   TP   ]  ← Bedrägerier som AI hanterade rätt / fel
```

- **TN (True Negative / Sann Negativ):** Normalt köp – AI sa Normalt ✓ (bra!)
- **TP (True Positive / Sann Positiv):** Bedrägeri – AI sa Bedrägeri ✓ (bra!)
- **FP (False Positive / Falsk Positiv):** Normalt köp – AI sa Bedrägeri ✗ (jobbigt för kunden)
- **FN (False Negative / Falsk Negativ):** Bedrägeri – AI sa Normalt ✗ (**farligt!** tjuven slipper undan)

**Vilken typ av fel är värst för en bank?** Tänk på det!

---
## 📊 Del 7 – Räkna på bedrägerieresultaten

In [ ]:
# Analysera resultaten
total_bedragerier_test = (y_test == 1).sum()
hittade_bedragerier = ((y_pred == 1) & (y_test == 1)).sum()
missade_bedragerier = ((y_pred == 0) & (y_test == 1)).sum()
falska_larm = ((y_pred == 1) & (y_test == 0)).sum()

print(f"Totalt antal bedrägerier i testdatan: {total_bedragerier_test}")
print()
print(f"✅ Hittade bedrägerier:  {hittade_bedragerier} ({hittade_bedragerier/total_bedragerier_test:.1%})")
print(f"❌ Missade bedrägerier:  {missade_bedragerier} ({missade_bedragerier/total_bedragerier_test:.1%})")
print(f"⚠️  Falska larm:          {falska_larm} (normala köp som flaggades)")

---
## 💬 Del 8 – Stor diskussion: Vad tycker du?

Nu när du sett resultaten, diskutera dessa frågor:

### Fråga 1: Vad är värst?
Tänk dig att du är chef på en bank. Vad är värst:
- **Alternativ A:** AI:n stoppar en riktig kunds köp av misstag *(falsk positiv)*  
  → Kunden blir irriterad men ingen skadar sig ekonomiskt
- **Alternativ B:** AI:n missar ett bedrägeri *(falsk negativ)*  
  → Tjuven lyckas stjäla pengar från ett konto

**Diskutera:** Kan det finnas situationer där svar A är värre? T.ex. vad händer om AI:n stoppar en sjuklings köp av medicin?

---

### Fråga 2: Hur löser man obalansproblemet?
AI-ingenjörer har flera tekniker för obalanserad data:
- **Oversampling:** Skapa fler konstgjorda bedrägerietransaktioner i träningsdatan
- **Undersampling:** Använd färre normala transaktioner i träningen
- **Vikta felen:** Berätta för modellen att missa ett bedrägeri är 100× värre än ett falskt larm

**Fråga:** Vilken metod verkar bäst för dig? Kan du se nackdelar med någon av dem?

---

### Fråga 3: Etik och rättvisa (Fairness)
Tänk om AI-modellen flaggar bedrägeri oftare för köp från specifika länder eller butiker.  
Är det okej?

**Fråga:** Vad är skillnaden mellan att modellen **hittar verkliga mönster** och att den är **diskriminerande**?

---
## 💡 Del 9 – Sammanfattning av hela kursen

Grattis! 🎉 Du har nu gått igenom alla 5 lektioner!

### Vad du lärt dig:

| Begrepp | Förklaring | Engelskt namn |
|---------|------------|---------------|
| **Övervakad inlärning** | AI lär sig från märkta exempel | Supervised Learning |
| **Egenskaper** | Det AI:n mäter/analyserar | Features |
| **Målvariabel** | Det AI:n ska förutsäga | Target / Label |
| **Träningsdata** | Data AI:n lär sig från | Training Data |
| **Testdata** | Data AI:n utvärderas på | Test Data |
| **Modell** | AI:ns "hjärna" | Model |
| **Beslutsträd** | Modell som ställer ja/nej-frågor | Decision Tree |
| **Noggrannhet** | Andel rätta svar | Accuracy |
| **Förväxlingsmatris** | Rutnät som visar vilka fel som gjordes | Confusion Matrix |
| **Obalanserad data** | En kategori är mycket vanligare | Imbalanced Data |
| **Falsk negativ** | Missad detektion (farligt!) | False Negative |
| **Falsk positiv** | Falskt larm | False Positive |

### Den stora insikten 🌟

Maskininlärning handlar inte bara om att skriva kod –  
det handlar om att **förstå problemet**, **välja rätt mått** och **fatta etiska beslut**.

AI är ett kraftfullt verktyg, men det är alltid **människor** som bestämmer hur det används!